# Knowledge Graph Analysis

Analyze paper relationships, concepts, and citation patterns using the Hive API.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

BASE = "http://localhost:7777"

def api(method, path, data=None):
    url = f"{BASE}{path}"
    r = requests.get(url) if method == "GET" else requests.post(url, json=data or {})
    return r.json()

## 1. Graph Overview

In [ ]:
graph = api("GET", "/api/graph")
nodes = pd.DataFrame(graph.get("nodes", []))
links = pd.DataFrame(graph.get("links", []))

print(f"Nodes: {len(nodes)}, Edges: {len(links)}")
if not nodes.empty:
    print("\nNode types:")
    display(nodes['type'].value_counts())

## 2. Node Type Distribution

In [ ]:
if not nodes.empty and 'type' in nodes.columns:
    nodes['type'].value_counts().plot(kind='bar', color='#60a5fa')
    plt.title('Knowledge Graph Node Types')
    plt.xlabel('Type')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()

## 3. Top Connected Concepts

In [ ]:
if not links.empty:
    print("Top relations:")
    display(links['relation'].value_counts().head(10))
    
    # Most connected nodes
    all_nodes = list(links['source']) + list(links['target'])
    top = pd.Series(all_nodes).value_counts().head(10)
    print("\nTop connected nodes:")
    display(top)

## 4. Paper Similarity Analysis

In [ ]:
sim = api("POST", "/api/similarity", {"algorithm": "abstract"})
df = pd.DataFrame(sim).sort_values('score', ascending=False)
print(f"Total pairs: {len(df)}")
if len(df) > 0:
    display(df[['source_title', 'target_title', 'score']].head(10))

## 5. Score Distribution

In [ ]:
if len(df) > 0:
    df['score'].hist(bins=20, color='#34d399', edgecolor='white')
    plt.title('Similarity Score Distribution')
    plt.xlabel('Score')
    plt.ylabel('Pairs')
    plt.show()